# Cervical Cancer Detection through Machine Learning-Based Classification Models Integrated with CNN

**Authors:** Samaher S. Alsharif & Shahd H. Altalhi

This notebook implements the project pipeline using the SIPaKMeD cervical-cell dataset. A CNN is trained as a deep feature extractor; PCA reduces the learned feature dimensionality; SVM and KNN classify cells into **Normal** and **Abnormal** groups.

## Project objective
The project explores automated Pap-smear image classification to support cervical cancer screening. The workflow includes dataset exploration, preprocessing, augmentation, CNN training, deep-feature extraction, PCA, hybrid CNN+SVM and CNN+KNN models, and evaluation with accuracy, classification reports, and confusion matrices.

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Conv2D, MaxPool2D, Flatten, Dense, Dropout
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.preprocessing import image

## 1. Dataset configuration
SIPaKMeD contains five cervical-cell categories: Superficial-Intermediate, Parabasal, Koilocytotic, Dyskeratotic, and Metaplastic. Update `base_dir` to the local dataset directory before running.

In [ ]:
base_dir = 'path/to/SIPaKMeD'
categories = [
    'im_Dyskeratotic',
    'im_Koilocytotic',
    'im_Metaplastic',
    'im_Parabasal',
    'im_Superficial-Intermediate'
]
normal_categories = ['im_Superficial-Intermediate', 'im_Parabasal']
abnormal_categories = ['im_Koilocytotic', 'im_Dyskeratotic', 'im_Metaplastic']

## 2. Exploratory Data Analysis

In [ ]:
image_counts = {}
for category in categories:
    category_path = os.path.join(base_dir, category, category, 'CROPPED')
    image_counts[category] = len([f for f in os.listdir(category_path) if f.lower().endswith(('.bmp','.jpg','.jpeg','.png'))])

counts_df = pd.DataFrame({'category': image_counts.keys(), 'count': image_counts.values()})
counts_df

In [ ]:
plt.figure(figsize=(10, 6))
sns.barplot(data=counts_df, x='category', y='count', hue='category', palette='viridis', legend=False)
plt.title('Count of Images in Each Category')
plt.xlabel('Category')
plt.ylabel('Number of Images')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 8))
plt.pie(image_counts.values(), labels=image_counts.keys(), autopct='%1.1f%%', startangle=140)
plt.title('Distribution of Images by Category')
plt.axis('equal')
plt.show()

In [ ]:
def display_images_from_folders(base_dir, categories, num_images=5):
    plt.figure(figsize=(15, 10))
    for i, category in enumerate(categories):
        category_path = os.path.join(base_dir, category, category, 'CROPPED')
        files = [f for f in os.listdir(category_path) if f.lower().endswith(('.bmp','.jpg','.jpeg','.png'))]
        for j, filename in enumerate(files[:num_images]):
            img = cv2.imread(os.path.join(category_path, filename))
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            plt.subplot(len(categories), num_images, i*num_images+j+1)
            plt.imshow(img)
            plt.axis('off')
            plt.title(category.replace('im_',''))
    plt.tight_layout()
    plt.show()

display_images_from_folders(base_dir, categories)

## 3. Build the binary classification table
The project groups **Superficial-Intermediate** and **Parabasal** as Normal, while **Koilocytotic**, **Dyskeratotic**, and **Metaplastic** are grouped as Abnormal.

In [ ]:
def create_image_dataframe(base_dir):
    rows = []
    for category in categories:
        category_path = os.path.join(base_dir, category, category, 'CROPPED')
        label = 'normal' if category in normal_categories else 'abnormal'
        for filename in os.listdir(category_path):
            if filename.lower().endswith(('.bmp','.jpg','.jpeg','.png')):
                rows.append({'image_path': os.path.join(category_path, filename), 'cell_type': category.replace('im_',''), 'label': label})
    return pd.DataFrame(rows)

df = create_image_dataframe(base_dir)
print(df.shape)
display(df.head())
print(df['label'].value_counts())

## 4. Image preprocessing
Images are resized to **64×64 RGB** and normalized to `[0,1]`. Labels are encoded numerically.

In [ ]:
def load_images_and_labels(df, target_size=(64,64)):
    images, labels = [], []
    for _, row in df.iterrows():
        img = cv2.imread(row['image_path'])
        img = cv2.resize(img, target_size)
        images.append(img)
        labels.append(row['label'])
    return np.array(images, dtype=np.float32)/255.0, np.array(labels)

images, labels = load_images_and_labels(df)
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(labels)
print('Images:', images.shape)
print('Classes:', list(label_encoder.classes_))

## 5. Train / validation / test split
The report uses **70% training, 15% validation, and 15% testing**.

In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(images, y, test_size=0.30, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp)
print('Train:', X_train.shape, 'Validation:', X_val.shape, 'Test:', X_test.shape)

## 6. Data augmentation
Training augmentation uses rotation, zoom, and horizontal flipping.

In [ ]:
datagen = ImageDataGenerator(rotation_range=30, zoom_range=0.2, horizontal_flip=True)
datagen.fit(X_train)

## 7. CNN model
The CNN learns image representations through two convolution/pooling blocks followed by a dense layer. Its penultimate representation is later used as input to SVM and KNN.

In [ ]:
cnn_model = Sequential([
    Input(shape=(64,64,3)),
    Conv2D(32,(3,3),activation='relu'),
    MaxPool2D((2,2)),
    Conv2D(64,(3,3),activation='relu'),
    MaxPool2D((2,2)),
    Flatten(),
    Dense(128,activation='relu'),
    Dropout(0.5),
    Dense(2,activation='softmax')
])
cnn_model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
cnn_model.summary()

In [ ]:
history = cnn_model.fit(datagen.flow(X_train, y_train, batch_size=32), epochs=10, validation_data=(X_val, y_val))

In [ ]:
plt.figure(figsize=(8,5))
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('CNN Training History')
plt.legend()
plt.show()

## 8. CNN feature extraction and PCA
The final softmax layer is removed so the CNN acts as a feature extractor. PCA reduces the learned representation to **50 components** before classical ML classification.

In [ ]:
feature_extractor = Sequential(cnn_model.layers[:-1])
train_features = feature_extractor.predict(X_train)
val_features = feature_extractor.predict(X_val)
test_features = feature_extractor.predict(X_test)

pca = PCA(n_components=50)
train_pca = pca.fit_transform(train_features)
val_pca = pca.transform(val_features)
test_pca = pca.transform(test_features)
print('PCA feature shape:', train_pca.shape)

## 9. CNN + SVM

In [ ]:
svm_model = SVC(kernel='linear', C=1.0)
svm_model.fit(train_pca, y_train)
svm_val_pred = svm_model.predict(val_pca)
svm_test_pred = svm_model.predict(test_pca)
svm_accuracy = accuracy_score(y_test, svm_test_pred)*100
print(f'Validation Accuracy: {accuracy_score(y_val, svm_val_pred)*100:.2f}%')
print(f'CNN + SVM Test Accuracy: {svm_accuracy:.2f}%')
print(classification_report(y_test, svm_test_pred, target_names=label_encoder.classes_))

In [ ]:
cm_svm = confusion_matrix(y_test, svm_test_pred)
sns.heatmap(cm_svm, annot=True, fmt='d', cmap='Blues', xticklabels=label_encoder.classes_, yticklabels=label_encoder.classes_)
plt.title('Confusion Matrix — CNN + SVM')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

## 10. CNN + KNN

In [ ]:
knn_model = KNeighborsClassifier(n_neighbors=5)
knn_model.fit(train_pca, y_train)
knn_val_pred = knn_model.predict(val_pca)
knn_test_pred = knn_model.predict(test_pca)
knn_accuracy = accuracy_score(y_test, knn_test_pred)*100
print(f'Validation Accuracy: {accuracy_score(y_val, knn_val_pred)*100:.2f}%')
print(f'CNN + KNN Test Accuracy: {knn_accuracy:.2f}%')
print(classification_report(y_test, knn_test_pred, target_names=label_encoder.classes_))

In [ ]:
cm_knn = confusion_matrix(y_test, knn_test_pred)
sns.heatmap(cm_knn, annot=True, fmt='d', cmap='Greens', xticklabels=label_encoder.classes_, yticklabels=label_encoder.classes_)
plt.title('Confusion Matrix — CNN + KNN')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

## 11. Model comparison
In the submitted project results, **CNN + SVM achieved 91.61% test accuracy** and **CNN + KNN achieved 94.57%**, making CNN + KNN the stronger of the two evaluated hybrid pipelines.

In [ ]:
results = pd.DataFrame({'Model':['CNN + SVM','CNN + KNN'], 'Test Accuracy (%)':[svm_accuracy,knn_accuracy]})
results.sort_values('Test Accuracy (%)', ascending=False)

## 12. Predict a new image
These helpers reproduce the project inference flow: preprocess → CNN features → PCA → classifier. Because the trained target is binary, the prediction returned is Normal or Abnormal.

In [ ]:
def preprocess_single_image(path):
    img = image.load_img(path, target_size=(64,64))
    arr = image.img_to_array(img)/255.0
    return np.expand_dims(arr, axis=0)

def predict_with_svm(path):
    arr = preprocess_single_image(path)
    features = feature_extractor.predict(arr)
    pred = svm_model.predict(pca.transform(features))[0]
    return label_encoder.inverse_transform([pred])[0]

def predict_with_knn(path):
    arr = preprocess_single_image(path)
    features = feature_extractor.predict(arr)
    pred = knn_model.predict(pca.transform(features))[0]
    return label_encoder.inverse_transform([pred])[0]

# Example:
# print(predict_with_svm('path/to/image.bmp'))
# print(predict_with_knn('path/to/image.bmp'))

## Key findings
- The CNN provides learned deep image features rather than relying on manually engineered features.
- PCA compresses the CNN representation before classical classification.
- Both SVM and KNN can operate on the CNN-derived features.
- In the submitted experiment, CNN+KNN produced the higher test accuracy (94.57%) compared with CNN+SVM (91.61%).

## Authors
**Samaher S. Alsharif**  
**Shahd H. Altalhi**